In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor


In [ ]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [ ]:
covariates = ["dolvol_lag2", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "cashpr", "chinv", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"]

I. Fonctions : normalisation et gestion des NaN

In [ ]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=306):
    """
    Génère des splits temporels selon la logique décrite :
    - Train cumulatif (augmente d'un an à chaque refit)
    - Validation = fenêtre fixe glissante de 1 an 
    - Test = fenêtre fixe après la validation
    - Avance de step_months à chaque itération : 12 mois

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop si on n'a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Avancer d'un step (ex : 12 mois) pour le prochain refit
        start += step_months

    return splits

In [ ]:
def preprocess_split(X_train, X_val, X_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).

    Paramètres
    ----------
    X_train, X_val, X_test : DataFrames bruts (avec 'Ticker' et 'Date')
    covariates : liste des colonnes numériques à traiter

    Retour
    ------
    X_train_scaled, X_val_scaled, X_test_scaled : DataFrames transformés (covariates seulement)
    """
    #Gestion des NaN : moyenne par Ticker calculée sur le train
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for col in covariates:
            #On remplace NaN par la moyenne du ticker
            df[col] = df.apply(
                lambda row: means_by_ticker[col][row["Ticker"]] 
                            if pd.isna(row[col]) and row["Ticker"] in means_by_ticker.index 
                            else row[col],
                axis=1
            )
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [ ]:
#Mesures : 
#R²
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

In [ ]:
#Permet de récupérer x et y 
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target]) #garde toutes les colonnes mais enlève excess return
    y = subset[target]
    return x, y

In [ ]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

BENCHMARK

In [ ]:
#HISTORICAL AVERAGE
r2_in_sample_ha = []
r2_oos_ha = []
success_ratio_in_ha = []
success_ratio_oos_ha = []

y_true = []
y_trainval_true = []

y_pred_ha = []
y_trainval_pred_ha = []   

dates_ha = []
tickers_ha = []
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits, start=1):

    tickers_trainval = pd.concat([x_train['Ticker'], x_val['Ticker']], ignore_index=True)
    y_trainval_all = pd.concat([y_train, y_val], ignore_index=True)
    trainval = pd.DataFrame({'Ticker': tickers_trainval, 'y': y_trainval_all})

    mean_by_ticker = trainval.groupby('Ticker')['y'].mean()

    preds_trainval = trainval['Ticker'].map(mean_by_ticker).values
    y_trainval_true.extend(trainval['y'].values)
    y_trainval_pred_ha.extend(preds_trainval)

    r2_in = r2(trainval['y'].values, preds_trainval)
    sr_in = success_ratio(trainval['y'].values, preds_trainval)

    r2_in_sample_ha.append(r2_in)
    success_ratio_in_ha.append(sr_in)

    preds_split = [mean_by_ticker.get(tkr, np.nan) for tkr in x_test['Ticker']]
    preds_split = np.array(preds_split)
    r2_out = r2(y_test, preds_split)
    sr_out = success_ratio(y_test, preds_split)

    r2_oos_ha.append(r2_out)
    success_ratio_oos_ha.append(sr_out)

    y_pred_ha.extend(preds_split)
    y_true.extend(y_test)
    dates_ha.append(x_test['Date'])
    tickers_ha.append(x_test['Ticker'])

    print(f"[Split {split_idx}] R² HA IN-sample: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

dates_ha = np.concatenate(dates_ha)
tickers_ha = np.concatenate(tickers_ha)
y_pred_ha = np.array(y_pred_ha)
y_true = np.array(y_true)

df_results_ha = pd.DataFrame({
    "Date": dates_ha,
    "Ticker": tickers_ha,
    "y_pred_ha": y_pred_ha,
    "y_true" : y_true
})

y_trainval_true = np.array(y_trainval_true)
y_trainval_pred_ha = np.array(y_trainval_pred_ha)

r2_in_ha_global = r2(y_trainval_true, y_trainval_pred_ha)
r2_oos_ha_global = r2(y_true, y_pred_ha)

ALGORITHMES

In [ ]:
"""
OLS : Ordinary Least Squares : Nous détaillons ici cet algorithme, la logique étant identique pour les autres modèles.
Listes utilisées : 
- r2_in_sample_list et r2_test_list : stockent, pour chaque split, les R² in‑sample et out‑of‑sample. Elles servent à analyser
  la performance split par split et à ajuster le tuning des modèles (utile pour les modèles à hyperparamètres).
- y_true : valeurs réelles de l’equity premium sur l’ensemble.
- y_pred_ols : prédictions correspondantes du modèle OLS. 
- y_trainval : données d’entraînement (train + validation) utilisées pour l’ajustement du modèle.
- dates_ols et tickers_ols : récupérées à chaque split pour pouvoir fusionner correctement les prédictions de tous les modèles
  et s’assurer que les lignes (dates/tickers) correspondent, évitant tout mélange potentiel des prédictions.

Df et résultats en sortie : 
- df_results_ols : df contenant les prédictions du modèles ols ainsi que la date et le ticker correspondant. 
- r2_results : dictionnaire contenant le r2 ols in sample et oos
"""

#pour calculer les r² globaux 
y_trainval_pred_ols = []

#stocke les r² par split 
r2_in_split_ols = []
r2_oos_split_ols = []
y_pred_ols = []

#pour les portefeuilles (voir notebook results)
dates_ols = []
tickers_ols = []

#sucess ratio 
success_ratio_in_ols = []
success_ratio_oos_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    #R² in-sample
    y_trainval_pred = ols.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_ols.append(r2_in)

    #R² oos
    y_test_pred = ols.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_ols.append(r2_out)

    #Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    #On stocke tout dans un tableau
    tickers_ols.append(x_test["Ticker"])
    dates_ols.append(x_test["Date"])
    y_pred_ols.append(y_test_pred)
    y_trainval_pred_ols.append(y_trainval_pred) 

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")
    
#On concatène les résultats de tous les splits en un df
dates_ols = np.concatenate(dates_ols)
tickers_ols = np.concatenate(tickers_ols)
y_pred_ols = np.concatenate(y_pred_ols)

y_trainval_pred_ols = np.concatenate(y_trainval_pred_ols)

df_results_ols = pd.DataFrame({
    "Date": dates_ols,
    "Ticker": tickers_ols, 
    "y_pred_ols": y_pred_ols,
})

print(f"\nMoyenne Success ratio in-sample OLS : {np.mean(success_ratio_in_ols):.6f}")
print(f"Moyenne Success ratio oos OLS : {np.mean(success_ratio_oos_ols):.6f}")

In [ ]:
"""
PLS : Partial Least Squares
Hyperparamètres :
- k : nombre de composantes latentes, choisi pour minimiser la MSE sur la validation.
"""

# Pour calculer les R² globaux
y_trainval_pred_pls = []

# Stocke les R² par split
r2_in_split_pls = []
r2_oos_split_pls = []
y_true_pls = []
y_pred_pls = []

# Pour les portefeuilles (voir notebook results)
dates_pls = []
tickers_pls = []

# Success ratio
success_ratio_in_pls = []
success_ratio_oos_pls = []

# Hyperparamètres PLS
best_components_list = []
mse_val_grids = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids.append(mse_val_grid)
    best_components_list.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_pls.append(r2_in)

    # R² oos
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_pls.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    # Stockage pour global
    tickers_pls.append(x_test["Ticker"])
    dates_pls.append(x_test["Date"])
    y_true_pls.append(y_test)
    y_pred_pls.append(y_test_pred)
    y_trainval_pred_pls.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_pls = np.concatenate(dates_pls)
tickers_pls = np.concatenate(tickers_pls)
y_true_pls = np.concatenate(y_true_pls)
y_pred_pls = np.concatenate(y_pred_pls)

y_trainval_pred_pls = np.concatenate(y_trainval_pred_pls)

df_results_pls = pd.DataFrame({
    "Date": dates_pls,
    "Ticker": tickers_pls,
    "y_true": y_true_pls,
    "y_pred_pls": y_pred_pls,
})

# Affichages
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_pls):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_pls):.6f}")

print(f"\nMoyenne Success ratio in-sample PLS : {np.mean(success_ratio_in_pls):.6f}")
print(f"Moyenne Success ratio oos PLS : {np.mean(success_ratio_oos_pls):.6f}")

In [ ]:
from sklearn.pipeline import Pipeline

"""
PCR : Principal Component Regression
Hyperparamètre : k (nombre de composantes principales)
"""

# Pour calculer les R² globaux
y_trainval_pred_pcr = []

# Stocke les R² par split
r2_in_split_pcr = []
r2_oos_split_pcr = []
y_true_pcr = []
y_pred_pcr = []

# Pour les portefeuilles (voir notebook results)
dates_pcr = []
tickers_pcr = []

# Success ratio
success_ratio_in_pcr = []
success_ratio_oos_pcr = []

# Hyperparamètres spécifiques PCR
best_components_pcr = []
mse_val_grids_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    candidate_ks = np.arange(1, 28)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    # Recherche du meilleur k
    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)

        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)
    print(f"Split {split_idx} : meilleur nombre de composantes k = {best_k}")

    # Réentraîner sur train+val avec le meilleur k
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_pcr.append(r2_in)

    # R² oos
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_pcr.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    # Stockage pour global
    tickers_pcr.append(x_test["Ticker"])
    dates_pcr.append(x_test["Date"])
    y_true_pcr.append(y_test)
    y_pred_pcr.append(y_test_pred)
    y_trainval_pred_pcr.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
tickers_pcr = np.concatenate(tickers_pcr)
dates_pcr = np.concatenate(dates_pcr)
y_true_pcr = np.concatenate(y_true_pcr)
y_pred_pcr = np.concatenate(y_pred_pcr)
y_trainval_pred_pcr = np.concatenate(y_trainval_pred_pcr)

df_results_pcr = pd.DataFrame({
    "Date": dates_pcr,
    "Ticker": tickers_pcr,
    "y_true": y_true_pcr,
    "y_pred_pcr": y_pred_pcr,
})

# Affichages
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_pcr):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_pcr):.6f}")

print(f"\nMoyenne Success ratio in-sample PCR : {np.mean(success_ratio_in_pcr):.6f}")
print(f"Moyenne Success ratio oos PCR : {np.mean(success_ratio_oos_pcr):.6f}")

In [ ]:
"""
ENet : Elastic Net

Hyperparamètres :
- lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
- l1_ratio fixé à 0.5
"""

# Pour calculer les R² globaux
y_trainval_pred_en = []

# Stocke les R² par split
r2_in_split_en = []
r2_oos_split_en = []
y_true_en = []
y_pred_en = []

# Pour les portefeuilles
dates_en = []
tickers_en = []

# Success ratio
success_ratio_in_en = []
success_ratio_oos_en = []

# Hyperparamètres spécifiques
best_lambdas = []

enet_param_grid = {
    'alpha': np.logspace(np.log10(0.0008), np.log10(0.0005), num=12)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Réentraîner sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = en_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_en.append(r2_in)

    # R² oos
    y_test_pred = en_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_en.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)

    # Stockage pour global
    tickers_en.append(x_test["Ticker"])
    dates_en.append(x_test["Date"])
    y_true_en.append(y_test)
    y_pred_en.append(y_test_pred)
    y_trainval_pred_en.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_en = np.concatenate(dates_en)
tickers_en = np.concatenate(tickers_en)
y_true_en = np.concatenate(y_true_en)
y_pred_en = np.concatenate(y_pred_en)
y_trainval_pred_en = np.concatenate(y_trainval_pred_en)

df_results_en = pd.DataFrame({
    "Date": dates_en,
    "Ticker": tickers_en,
    "y_true": y_true_en,
    "y_pred_en": y_pred_en,
})

# Affichages
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_en):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_en):.6f}")

print(f"\nMoyenne Success ratio in-sample ENet : {np.mean(success_ratio_in_en):.6f}")
print(f"Moyenne Success ratio oos ENet : {np.mean(success_ratio_oos_en):.6f}")


In [ ]:
"""
RF : Random Forest
Hyperparamètres :
- n_estimators : nombre d’arbres dans la forêt.
- max_depth : profondeur maximale de chaque arbre.
- min_samples_leaf : nombre minimal d’échantillons dans une feuille.
- max_features : nombre de variables considérées pour le split.
"""

param_grid_rf = {
    'n_estimators': [100, 125, 150],
    'max_depth': [5, 6, 7],
    'min_samples_leaf': [1, 2, 3],
    'max_features': ['log2', None]
}

# Pour calculer les R² globaux
y_trainval_pred_rf = []

# Stocke les R² par split
r2_in_split_rf = []
r2_oos_split_rf = []
y_true_rf = []
y_pred_rf = []

# Pour les portefeuilles
dates_rf = []
tickers_rf = []

# Success ratio
success_ratio_in_rf = []
success_ratio_oos_rf = []

# Hyperparamètres spécifiques
best_params_rf = []
mse_val_grids_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(
            **params,
            n_jobs=-1,
            random_state=0
        )
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    rf_final = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = rf_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_rf.append(r2_in)

    # R² oos
    y_test_pred = rf_final.predict(x_test[covariates])
    r2_out = r2(y_test.values, y_test_pred)
    r2_oos_split_rf.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test.values, y_test_pred)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)

    # Stockage pour global
    tickers_rf.append(x_test["Ticker"])
    dates_rf.append(x_test["Date"])
    y_true_rf.append(y_test)
    y_pred_rf.append(y_test_pred)
    y_trainval_pred_rf.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_rf = np.concatenate(dates_rf)
tickers_rf = np.concatenate(tickers_rf)
y_true_rf = np.concatenate(y_true_rf)
y_pred_rf = np.concatenate(y_pred_rf)
y_trainval_pred_rf = np.concatenate(y_trainval_pred_rf)

df_results_rf = pd.DataFrame({
    "Date": dates_rf,
    "Ticker": tickers_rf,
    "y_true": y_true_rf,
    "y_pred_rf": y_pred_rf,
})

In [ ]:
"""
GBRT : Gradient Boosted Regression Trees
Hyperparamètres :
- n_estimators : nombre d’arbres successifs
- learning_rate : taux d’apprentissage
- max_depth : profondeur maximale des arbres
- loss : fonction de perte
- alpha : paramètre huber
"""
param_grid_gbrt = {
    'n_estimators': [200, 300],       
    'learning_rate': [0.005, 0.01], 
    'max_depth': [2, 3, 4],              
    'loss': ['huber'],
    'alpha': [0.9]
}


# Pour calculer les R² globaux
y_trainval_pred_gbrt = []

# Stocke les R² par split
r2_in_split_gbrt = []
r2_oos_split_gbrt = []
y_true_gbrt = []
y_pred_gbrt = []

# Pour les portefeuilles
dates_gbrt = []
tickers_gbrt = []

# Success ratio
success_ratio_in_gbrt = []
success_ratio_oos_gbrt = []

# Hyperparamètres spécifiques
best_params_gbrt = []
mse_val_grids_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_gbrt):
        gbrt = GradientBoostingRegressor(**params, random_state=0)
        gbrt.fit(x_train[covariates], y_train)
        y_val_pred = gbrt.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = gbrt_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_gbrt.append(r2_in)

    # R² oos
    y_test_pred = gbrt_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_gbrt.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Stockage pour global
    tickers_gbrt.append(x_test["Ticker"])
    dates_gbrt.append(x_test["Date"])
    y_true_gbrt.append(y_test)
    y_pred_gbrt.append(y_test_pred)
    y_trainval_pred_gbrt.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_gbrt = np.concatenate(dates_gbrt)
tickers_gbrt = np.concatenate(tickers_gbrt)
y_true_gbrt = np.concatenate(y_true_gbrt)
y_pred_gbrt = np.concatenate(y_pred_gbrt)
y_trainval_pred_gbrt = np.concatenate(y_trainval_pred_gbrt)

df_results_gbrt = pd.DataFrame({
    "Date": dates_gbrt,
    "Ticker": tickers_gbrt,
    "y_true": y_true_gbrt,
    "y_pred_gbrt": y_pred_gbrt,
})

In [ ]:
"""
XGB : XGBoost Regressor
Hyperparamètres :
- n_estimators : nombre d’arbres
- max_depth : profondeur max
- eta : learning rate
"""

param_grid_xgb = {
    'n_estimators': [150, 200, 250],
    'max_depth': [3, 4],
    'eta': [0.01, 0.02],
}

# Pour calculer les R² globaux
y_trainval_pred_xgb = []

# Stocke les R² par split
r2_in_split_xgb = []
r2_oos_split_xgb = []
y_true_xgb = []
y_pred_xgb = []

# Pour les portefeuilles
dates_xgb = []
tickers_xgb = []

# Success ratio
success_ratio_in_xgb = []
success_ratio_oos_xgb = []

# Hyperparamètres spécifiques
best_params_xgb = []
mse_val_grids_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    # Grid search
    for params in ParameterGrid(param_grid_xgb):
        xgb_model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        xgb_model.fit(x_train[covariates], y_train)
        y_val_pred = xgb_model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_xgb.append(mse_grid)
    best_params_xgb.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    xgb_final = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    xgb_final.fit(x_trainval, y_trainval)

    # R² in-sample
    y_trainval_pred = xgb_final.predict(x_trainval)
    r2_in = r2(y_trainval.values, y_trainval_pred)
    r2_in_split_xgb.append(r2_in)

    # R² oos
    y_test_pred = xgb_final.predict(x_test[covariates])
    r2_out = r2(y_test, y_test_pred)
    r2_oos_split_xgb.append(r2_out)

    # Success ratio
    sr_in = success_ratio(y_trainval.values, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Stockage pour global
    tickers_xgb.append(x_test["Ticker"])
    dates_xgb.append(x_test["Date"])
    y_pred_xgb.append(y_test_pred)
    y_trainval_pred_xgb.append(y_trainval_pred)

    print(f"R² in-sample : {r2_in:.6f} | R² oos : {r2_out:.6f}")

# Concaténation des résultats
dates_xgb = np.concatenate(dates_xgb)
tickers_xgb = np.concatenate(tickers_xgb)
y_pred_xgb = np.concatenate(y_pred_xgb)
y_trainval_pred_xgb = np.concatenate(y_trainval_pred_xgb)

df_results_xgb = pd.DataFrame({
    "Date": dates_xgb,
    "Ticker": tickers_xgb,
    "y_pred_xgb": y_pred_xgb,
})

# Affichages
print(f"Moyenne des R² in-sample (splits) : {np.mean(r2_in_split_xgb):.6f}")
print(f"Moyenne des R² oos (splits) : {np.mean(r2_oos_split_xgb):.6f}")

print(f"\nMoyenne Success ratio in-sample XGB : {np.mean(success_ratio_in_xgb):.6f}")
print(f"Moyenne Success ratio oos XGB : {np.mean(success_ratio_oos_xgb):.6f}")


METRIQUES

In [ ]:
#Calculs métriques in-sample 

predictions_in = {
    "OLS" : y_trainval_pred_ols,
    "PLS" : y_trainval_pred_pls,
    "PCR" : y_trainval_pred_pcr,
    "Enet" : y_trainval_pred_en,
    "RF" : y_trainval_pred_rf,
    "GBRT" : y_trainval_pred_gbrt,
    "XGB" : y_trainval_pred_xgb,
    "HA" : y_trainval_pred_ha
}

#TABLE 1 ET 2 : R², success ratio, comparaison HA
rows = []
for model_name, y_pred in predictions_in.items():
    r2_vs = r2_vs_benchmark(y_trainval_true, y_pred, y_trainval_pred_ha)
    r2_in = r2(y_trainval_true, y_pred)
    sr_in = success_ratio(y_trainval_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "In-sample $R^2$": r2_in,
        "Success Ratio": sr_in,
        "$R^2$ vs HA" : r2_vs })
    
df_r2_in_vs = pd.DataFrame(rows)

# Pivot simple pour transformer les splits en colonnes
df_pivot = df_r2_splits_in.pivot(index="Model", columns="Split", values="R²_in")
df_pivot["Mean"] = df_pivot.mean(axis=1)
df_pivot = df_pivot[ [1, 2, 3, "Mean"] ]
order = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB"]
df_pivot = df_pivot.reindex(order)

#print(df_r2_in_vs)
#print(df_pivot)

##Conversion → Latex 
df_latex_r2in = df_r2_in_vs.copy()
cols_to_convert = ["$R^2$ vs HA", "In-sample $R^2$", "Success Ratio"]

for col in cols_to_convert:
    df_latex_r2in[col] = (df_latex_r2in[col] * 100).apply(lambda x: f"{x:.2f}") #* 100 et arrondir trois chiffres après la virgule

latex_table = df_latex_r2in.to_latex(index=False, escape=False)
print(latex_table)


In [ ]:
#Calculs métriques out-of-sample : R² moyen, R² par split 

predictions_oos = {
    "OLS" : y_pred_ols,
    "PLS" : y_pred_pls,
    "PCR" : y_pred_pcr,
    "Enet" : y_pred_en,
    "RF" : y_pred_rf,
    "GBRT" : y_pred_gbrt,
    "XGB" : y_pred_xgb,
    "HA" : y_pred_ha
}

all_splits_oos = {
    "OLS": r2_oos_split_ols,
    "PLS": r2_oos_split_pls,
    "PCR": r2_oos_split_pcr,
    "Enet": r2_oos_split_en,
    "RF": r2_oos_split_rf,
    "GBRT": r2_oos_split_gbrt,
    "XGB": r2_oos_split_xgb,
    #"HA": r2_oos_split_ha
}

rows = []
#récupérer les R²
for model_name, y_pred in predictions_oos.items():
    r2_oos = r2(y_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "Out-of-sample $R^2$": r2_oos,})
df_r2_oos = pd.DataFrame(rows)

rows_splits = []  
for model_name, values in all_splits_oos.items():  # <--- itère aussi sur la liste
    row = {"Model": model_name}
    for i, val in enumerate(values, start=1):
        row[f"R² Split {i}"] = val
    rows_splits.append(row)

df_splits = pd.DataFrame(rows_splits)


df_r2_oos = pd.merge(df_splits, df_r2_oos, on="Model", how="left") 

#vers latex
df_r2_oos_latex = df_r2_oos.copy()
for col in df_r2_oos_latex.columns:
    if col != "Model":  # garde la colonne texte telle quelle
        df_r2_oos_latex[col] = (df_r2_oos_latex[col] * 100).apply(lambda x: f"{x:.2f}")
latex_table = df_r2_oos_latex.to_latex(index=False, escape=False)
print(latex_table)


In [ ]:
TABLE 1 ET 2 : R², success ratio, comparaison HA
rows = []
for model_name, y_pred in predictions_in.items():
    r2_vs = r2_vs_benchmark(y_trainval_true, y_pred, y_trainval_pred_ha)
    r2_in = r2(y_trainval_true, y_pred)
    sr_in = success_ratio(y_trainval_true, y_pred)
    rows.append({
        "Model" : model_name, 
        "In-sample $R^2$": r2_in,
        "Success Ratio": sr_in,
        "$R^2$ vs HA" : r2_vs })
    
df_r2_in_vs = pd.DataFrame(rows)


#Table 3 : Split R² 
all_splits = {
    "OLS": r2_in_split_ols,
    "PLS": r2_in_split_pls,
    "PCR": r2_in_split_pcr,
    "Enet": r2_in_split_en,
    "RF": r2_in_split_rf,
    "GBRT": r2_in_split_gbrt,
    "XGB": r2_in_split_xgb
}

rows = []
for model, values in all_splits.items():
    for i, val in enumerate(values, start=1):
        rows.append({"Model": model, "Split": i, "R²_in": val})
    

df_r2_splits_in = pd.DataFrame(rows)

# Pivot simple pour transformer les splits en colonnes
df_pivot = df_r2_splits_in.pivot(index="Model", columns="Split", values="R²_in")
df_pivot["Mean"] = df_pivot.mean(axis=1)
df_pivot = df_pivot[ [1, 2, 3, "Mean"] ]
order = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB"]
df_pivot = df_pivot.reindex(order)

#print(df_r2_in_vs)
#print(df_pivot)

##Conversion → Latex 
df_latex_r2in = df_r2_in_vs.copy()
cols_to_convert = ["$R^2$ vs HA", "In-sample $R^2$", "Success Ratio"]

for col in cols_to_convert:
    df_latex_r2in[col] = (df_latex_r2in[col] * 100).apply(lambda x: f"{x:.2f}") #* 100 et arrondir trois chiffres après la virgule

latex_table = df_latex_r2in.to_latex(index=False, escape=False)
print(latex_table)


In [ ]:
#DIABEDOLD TEST 
import numpy as np
import pandas as pd
from scipy.stats import norm

# ----------------------------
# Hypothèse : y_true est un array/Series de shape (N,)
# predictions_oos est un dict {"OLS": y_pred_ols, "GBRT": y_pred_gbrt, ...}
# Chaque y_pred_* est un array/Series de shape (N,) aligné avec y_true
# ----------------------------

# Fonction pour calculer la statistique DM entre deux vecteurs de prédictions
def dm_test(y_true, y_pred_1, y_pred_2):
    # erreurs
    e1 = y_true - y_pred_1
    e2 = y_true - y_pred_2
    # différences de pertes
    d_t = (e1**2 - e2**2)
    mean_d = d_t.mean()
    # variance avec hypothèse d'indépendance (simple)
    # (si tu veux une HAC Newey-West, je peux te coder ça aussi)
    var_d = d_t.var(ddof=1) / len(d_t)
    se_d = np.sqrt(var_d)
    dm_stat = mean_d / se_d if se_d > 0 else np.nan
    return dm_stat

# ----------------------------
# Construire la matrice DM
# ----------------------------
model_names = list(predictions_oos.keys())
dm_matrix = pd.DataFrame(np.zeros((len(model_names), len(model_names))),
                         index=model_names, columns=model_names)

alpha = 0.05
z_crit = norm.ppf(1 - alpha/2)  # seuil 5%

for i, m_row in enumerate(model_names):
    for j, m_col in enumerate(model_names):
        if i == j:
            dm_matrix.iloc[i, j] = np.nan
        else:
            stat = dm_test(y_true,
                           predictions_oos[m_row],
                           predictions_oos[m_col])
            # Ajouter astérisque si significatif
            mark = ""
            if abs(stat) > z_crit:
                mark = "*"
            # Arrondir et concaténer
            dm_matrix.iloc[i, j] = f"{stat:.2f}{mark}"

# ----------------------------
# Résultat : dm_matrix est un DataFrame comme dans ton image
# ----------------------------
print(dm_matrix)

# Pour exporter en LaTeX :
latex_table = dm_matrix.to_latex(index=True, escape=False)
print(latex_table)


PORTEFEUILLES

In [ ]:
#prédictions : all model 
df_predict = df_results_ols[["Date", "Ticker"]].copy()
df_predict = df_predict.merge(df_results_pls, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_pcr, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_en, on=["Date", "Ticker"], how="inner")
df_predict = df_predict.merge(df_results_rf, on=["Date", "Ticker"], how="inner")

III. RESULTATS

In [ ]:
#TABLE 4: R² in sample comparé au HA benchmark 
predictions_oos = {
    "OLS" : y_pred_ols,
    "PLS" : y_pred_pls,
    "PCR" : y_pred_pcr,
    "Enet" : y_pred_en,
    "RF" : y_pred_rf,
    "GBRT" : y_pred_gbrt,
    #"XGB" : y_pred_xgb
}

for model_name, y_pred in predictions_oos.items():
    r2_vs = r2_vs_benchmark(y_true, y_pred, y_pred_ha)
    rows.append({"Model" : model_name, "R² vs HA" : r2_vs })

df_r2_oos_vs = pd.DataFrame(rows)
print(df_r2_oos_vs)

In [ ]:
#PREMIER TABLEAU : R² OOS Monthly

#* 100 pcq on est en décimal 
R2_monthly_sum = {
    'OLS' : df_r2_monthly["R2_OLS"].mean() * 100,
    'PLS' : df_r2_monthly_pls["R2_PLS"].mean() * 100,
    'PCR' : df_r2_monthly_pcr["R2_PCR"].mean() * 100, 
    'Enet' : df_r2_monthly_en["R2_ENET"].mean() * 100,
    'RF' : df_r2_monthly_rf["R2_RF"].mean() * 100,
    'GBRT' : df_r2_monthly_gbrt["R2_GBRT"].mean() * 100
}


#tableau 
# Transformer le dictionnaire en DataFrame pour un tableau
df_table = pd.DataFrame.from_dict(R2_monthly_sum, orient='index', columns=['Mean_R2_OOS'])
print("=== Tableau résumé R² OOS Monthly ===")
print(df_table)



In [ ]:
import matplotlib.pyplot as plt

models = list(R2_monthly_sum.keys())
values = list(R2_monthly_sum.values())

plt.figure(figsize=(8,5))
bars = plt.bar(models, values, edgecolor='black', width=0.35)  # width < 1 pour des barres plus fines

# Titre et labels
plt.title('Comparaison des modèles (moyenne mensuelle)', fontsize=14, fontweight='bold')
plt.ylabel('Mean monthly $R^2_{OOS}$ (%)', fontsize=12)

# Valeurs au-dessus des barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.05,
             f'{height:.2f}', ha='center', va='bottom', fontsize=10)

# Couleur personnalisée
for bar in bars:
    bar.set_color('#4C72B0')

# Style épuré
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
#R2 benchmark HA 

def compute_oos_r_square(actual, y_benchmark, y_pred):
    MSFE_benchmark = mean_squared_error(y_benchmark, actual)
    MSFE_pred = mean_squared_error(y_pred, actual)
    return 1 - MSFE_pred / MSFE_benchmark

In [ ]:
#HA Benchmark 
actual_test = all_y_true_ols          # identique pour tous les modèles
y_pred_HA = all_y_pred_HA

#Prédictions concaténées dans un dictionnaire
model_predictions = {
    "OLS": all_y_pred_ols,
    "PLS": all_y_pred_pls,
    "PCR": all_y_pred_pcr,
    "ENet": all_y_pred_en,
    "RF": all_y_pred_rf,
    "GBRT": all_y_pred_gbrt
}

#Calcul des métriques
results = []

for model_name, y_pred in model_predictions.items():
    r2_oos = compute_oos_r_square(actual_test, y_pred_HA, y_pred) * 100  # en %
    results.append([model_name, r2_oos])

results.append(["HA", 0.0])

#df final 
df_r2_oos = pd.DataFrame(results, columns=["Model", "OOS_R2(%)"])

print(df_r2_oos)